# 03 — Extracción de Features Espaciales

Extrae features ambientales y de presión humana para cada punto de ocurrencia GBIF.
Al final agrega todo a nivel especie → un DataFrame listo para modelar.

**Fuentes:**
| Feature | Fuente | Tipo |
|---|---|---|
| bio1,4,7,12,14,15 | WorldClim v2.1 | Raster |
| pct_forest_loss_total / recent | GFW lossyear | Raster |
| landcover | MapBiomas | Raster |
| pct/dist ANP | RAISG ANPs nacional+departamental | Vector |
| pct/dist TI | RAISG Territorios Indígenas titulados | Vector |
| pct/dist minería ilegal | RAISG MineriaIlegal_pol | Vector |
| pct/dist concesión minera | RAISG ZonasMineras en explotación | Vector |
| pct/dist petróleo | RAISG Petroleo | Vector |
| dist vía | RAISG Vias_nacional | Vector |
| quemas_freq | RAISG quemas 2020 | Vector |


## 0. Setup

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterstats import point_query
from shapely.geometry import box
from pathlib import Path
from tqdm.notebook import tqdm

DATA       = Path('../data')
RAISG      = DATA / 'raisg'
OUT        = DATA / 'processed'
OUT.mkdir(exist_ok=True)

# CRS proyectado para cálculos de distancia (UTM zona 20S — cubre Bolivia)
CRS_PROJ = 'EPSG:32720'

# Leyenda MapBiomas → categorías simplificadas
MAPBIOMAS_LEGEND = {
    0:  'no_data',
    3:  'forest',        # Formação Florestal
    4:  'forest',        # Formação Savânica
    6:  'natural',       # Floresta Alagável
    11: 'natural',       # Área Úmida não Florestal
    15: 'agriculture',   # Pastagem
    22: 'agriculture',   # Área não Vegetada
    26: 'water',         # Corpo d'Água
    42: 'natural',       # Formação Campestre
    43: 'natural',       # Apicum
    44: 'water',         # Manguezal (agua)
    45: 'agriculture',   # Soja
    57: 'natural',       # Savana Amazônica
    58: 'agriculture',   # Silvicultura
}

print('Setup OK')

Setup OK


## 1. Cargar ocurrencias y convertir a GeoDataFrame

In [2]:
occ = pd.read_csv(DATA / 'processed/occurrences.csv', low_memory=False)

# Filtro de especies domesticas
# Nota: Lama guanicoe y Vicugna vicugna son silvestres y NO se filtran.
DOMESTICAS = {
    "Ovis aries", "Equus caballus", "Bos taurus", "Capra hircus",
    "Bubalus bubalis", "Sus scrofa domesticus",
    "Canis lupus familiaris", "Felis catus",
    "Lama glama", "Vicugna pacos",       # llama y alpaca (domesticas)
    "Rattus norvegicus", "Mus musculus",  # ratas/ratones comensales
    "Gallus gallus",                      # gallina
    "Columba livia",                      # paloma domestica
}
antes = len(occ)
occ = occ[~occ['species'].isin(DOMESTICAS)]
print(f'Registros filtrados (domesticas): {antes - len(occ):,}')

# Filtrar: solo registros con coordenadas validas
occ = occ.dropna(subset=['decimalLatitude', 'decimalLongitude'])
occ = occ[occ['iucn_categoria'].isin(['LC', 'NT', 'VU', 'EN', 'CR']) | occ['iucn_categoria'].isna()]
# Excluir DD explicitamente
occ = occ[occ['iucn_categoria'] != 'DD']

gdf = gpd.GeoDataFrame(
    occ,
    geometry=gpd.points_from_xy(occ['decimalLongitude'], occ['decimalLatitude']),
    crs='EPSG:4326'
)

print(f'Ocurrencias: {len(gdf):,}')
print(f'Especies: {gdf["species"].nunique():,}')
print(f'Aves: {gdf[gdf["class"]=="Aves"]["species"].nunique():,} | '
      f'Mamiferos: {gdf[gdf["class"]=="Mammalia"]["species"].nunique():,}')


Registros filtrados (domesticas): 6,688
Ocurrencias: 1,365,972
Especies: 1,693
Aves: 1,476 | Mamiferos: 217


## 2. Extracción de rasters

 extrae el valor del píxel en cada punto de forma vectorizada (rápido con millones de puntos).

In [3]:
def extract_raster(gdf, raster_path, col_name):
    """Extrae valor de raster en cada punto usando rasterio.sample (vectorizado, rápido)."""
    coords = list(zip(gdf.geometry.x, gdf.geometry.y))
    with rasterio.open(raster_path) as src:
        vals = [v[0] for v in src.sample(coords)]
    return pd.Series(vals, index=gdf.index, name=col_name)

In [4]:
print('Extrayendo WorldClim...')
wc_dir = DATA / 'worldclim'
for bio in tqdm(['bio1', 'bio4', 'bio7', 'bio12', 'bio14', 'bio15']):
    gdf[bio] = extract_raster(gdf, wc_dir / f'{bio}_bolivia.tif', bio)

print('Extrayendo GFW lossyear...')
gdf['lossyear'] = extract_raster(gdf, DATA / 'gfw/lossyear_bolivia.tif', 'lossyear')
# 0 = sin pérdida, 1-23 = año de pérdida (2001-2023)
gdf['forest_loss_total']  = (gdf['lossyear'] > 0).astype(float)   # pérdida 2001-2023
gdf['forest_loss_recent'] = (gdf['lossyear'] >= 15).astype(float)  # pérdida 2015-2023

print('Extrayendo MapBiomas...')
gdf['lc_code'] = extract_raster(gdf, DATA / 'mapbiomas/mapbiomas_bolivia.tif', 'lc_code')
gdf['lc_code'] = gdf['lc_code'].fillna(0).astype(int)
gdf['lc_cat']  = gdf['lc_code'].map(MAPBIOMAS_LEGEND).fillna('no_data')

print('Rasters OK')
gdf[['bio1','bio12','lossyear','forest_loss_total','forest_loss_recent','lc_cat']].describe()

Extrayendo WorldClim...


  0%|          | 0/6 [00:00<?, ?it/s]

Extrayendo GFW lossyear...
Extrayendo MapBiomas...
Rasters OK


,bio1,bio12,lossyear,forest_loss_total,forest_loss_recent
count,1.365972e+06,1.365972e+06,1.365972e+06,1.365972e+06,1.365972e+06
mean,-inf,-inf,6.387920e-01,4.840290e-02,2.110219e-02
std,inf,inf,3.111542e+00,2.146161e-01,1.437251e-01
min,-3.400000e+38,-3.400000e+38,0.000000e+00,0.000000e+00,0.000000e+00
25%,1.761968e+01,8.150000e+02,0.000000e+00,0.000000e+00,0.000000e+00
50%,2.413335e+01,1.130000e+03,0.000000e+00,0.000000e+00,0.000000e+00
75%,2.537170e+01,1.684000e+03,0.000000e+00,0.000000e+00,0.000000e+00
max,2.711644e+01,4.038000e+03,2.300000e+01,1.000000e+00,1.000000e+00


## 3. Features vectoriales RAISG

Para cada capa vectorial calculamos:
- **pct_en_X**: si el punto cae dentro del polígono (1/0) → luego se agrega como % por especie
- **dist_X_km**: distancia al polígono más cercano en km

In [5]:
def add_containment_and_distance(gdf, polygons, col_prefix):
    """
    Agrega dos columnas al GeoDataFrame de puntos:
      {col_prefix}_inside : 1.0 si el punto está dentro de algún polígono, 0.0 si no
      {col_prefix}_dist_km: distancia en km al polígono más cercano
    """
    # Proyectar a CRS métrico para distancias
    gdf_proj    = gdf[['geometry']].to_crs(CRS_PROJ)
    polys_proj  = polygons[['geometry']].to_crs(CRS_PROJ)

    # Containment via sjoin
    joined = gpd.sjoin(gdf_proj, polys_proj, how='left', predicate='within')
    inside = ~joined['index_right'].isna()
    # En caso de duplicados por múltiples polígonos solapados, tomar OR
    inside = inside.groupby(level=0).any()
    gdf[f'{col_prefix}_inside'] = inside.reindex(gdf.index).fillna(False).astype(float)

    # Distancia al más cercano
    nearest = gpd.sjoin_nearest(gdf_proj, polys_proj, how='left', distance_col='dist_m')
    dist    = nearest['dist_m'].groupby(level=0).min()
    gdf[f'{col_prefix}_dist_km'] = (dist.reindex(gdf.index).fillna(np.nan) / 1000)

    return gdf

In [6]:
# --- 3.1 Áreas Naturales Protegidas (nacional + departamental) ---
print('Cargando ANPs...')
anp_nac  = gpd.read_file(RAISG / 'Anps_jun2025/ANP_nacional.shp').to_crs('EPSG:4326')
anp_dep  = gpd.read_file(RAISG / 'Anps_jun2025/ANP_departamental.shp').to_crs('EPSG:4326')

# Filtrar solo Bolivia y solo geometrías válidas
anp_nac  = anp_nac[anp_nac['pais'] == 'Bolivia'][['geometry']]
anp_dep  = anp_dep[anp_dep['pais'] == 'Bolivia'][['geometry']]
anp      = pd.concat([anp_nac, anp_dep], ignore_index=True)
anp      = anp[anp.geometry.is_valid]

print(f'  ANPs Bolivia: {len(anp)} polígonos')
gdf = add_containment_and_distance(gdf, anp, 'anp')
print(f'  Puntos dentro de ANP: {gdf["anp_inside"].mean()*100:.1f}%')

Cargando ANPs...
  ANPs Bolivia: 155 polígonos
  Puntos dentro de ANP: 30.9%


In [7]:
# --- 3.2 Territorios Indígenas (solo titulados) ---
print('Cargando TIs...')
tis = gpd.read_file(RAISG / 'Tis_Junho2025/Tis_TerritoriosIndigenas.shp').to_crs('EPSG:4326')
tis = tis[(tis['pais'] == 'Bolivia') & (tis['status'] == 'Titulada')][['geometry']]
tis = tis[tis.geometry.is_valid]

print(f'  TIs tituladas Bolivia: {len(tis)} territorios')
gdf = add_containment_and_distance(gdf, tis, 'ti')
print(f'  Puntos dentro de TI: {gdf["ti_inside"].mean()*100:.1f}%')

Cargando TIs...
  TIs tituladas Bolivia: 113 territorios
  Puntos dentro de TI: 5.2%


In [8]:
# --- 3.3 Minería ilegal (polígonos 2020) ---
print('Cargando minería ilegal...')
mil = gpd.read_file(RAISG / 'MIneriaIlegal_2020/MineriaIlegal_pol.shp').to_crs('EPSG:4326')
mil = mil[mil['país'] == 'Bolivia'][['geometry']]
mil = mil[mil.geometry.is_valid]

print(f'  Polígonos minería ilegal Bolivia: {len(mil)}')
gdf = add_containment_and_distance(gdf, mil, 'min_ilegal')
print(f'  Puntos dentro de minería ilegal: {gdf["min_ilegal_inside"].mean()*100:.2f}%')

Cargando minería ilegal...
  Polígonos minería ilegal Bolivia: 3
  Puntos dentro de minería ilegal: 2.11%


In [9]:
# --- 3.4 Concesiones mineras (en explotación) ---
print('Cargando zonas mineras...')
zm = gpd.read_file(RAISG / 'ZonasMineras_jun2025/mineria.shp').to_crs('EPSG:4326')
zm = zm[(zm['pais'] == 'Bolivia') & (zm['situacion'] == 'en explotación')][['geometry']]
zm = zm[zm.geometry.is_valid]

print(f'  Concesiones activas Bolivia: {len(zm)}')
gdf = add_containment_and_distance(gdf, zm, 'concesion')
print(f'  Puntos dentro de concesión activa: {gdf["concesion_inside"].mean()*100:.2f}%')

Cargando zonas mineras...
  Concesiones activas Bolivia: 1576
  Puntos dentro de concesión activa: 0.84%


In [10]:
# --- 3.5 Bloques petroleros ---
print('Cargando petróleo/gas...')
pet = gpd.read_file(RAISG / 'Petroleo_jun2025/petroleo.shp').to_crs('EPSG:4326')
pet = pet[pet['pais'] == 'Bolivia'][['geometry']]
pet = pet[pet.geometry.is_valid]

print(f'  Bloques petroleros Bolivia: {len(pet)}')
gdf = add_containment_and_distance(gdf, pet, 'petroleo')
print(f'  Puntos dentro de bloque petrolero: {gdf["petroleo_inside"].mean()*100:.1f}%')

Cargando petróleo/gas...
  Bloques petroleros Bolivia: 119
  Puntos dentro de bloque petrolero: 16.1%


In [11]:
# --- 3.6 Distancia a vías nacionales ---
print('Cargando vías nacionales...')
vias = gpd.read_file(RAISG / 'Vias_jun2025/vias_nacional.shp').to_crs('EPSG:4326')
vias = vias[vias['pais'] == 'Bolivia'][['geometry']]
vias = vias[vias.geometry.is_valid]

# Para líneas solo distancia, no containment
print(f'  Tramos viales Bolivia: {len(vias)}')
vias_proj = vias.to_crs(CRS_PROJ)
gdf_proj  = gdf[['geometry']].to_crs(CRS_PROJ)
nearest   = gpd.sjoin_nearest(gdf_proj, vias_proj, how='left', distance_col='dist_m')
dist_via  = nearest['dist_m'].groupby(level=0).min()
gdf['via_dist_km'] = (dist_via.reindex(gdf.index).fillna(np.nan) / 1000)
print(f'  Distancia media a vía: {gdf["via_dist_km"].mean():.1f} km')

Cargando vías nacionales...
  Tramos viales Bolivia: 197
  Distancia media a vía: 19.4 km


In [12]:
# --- 3.7 Frecuencia de quemas ---
print('Cargando quemas...')
quemas = gpd.read_file(RAISG / 'quemas2020/quemas.shp').to_crs('EPSG:4326')
quemas = quemas[quemas.geometry.is_valid][['geometry', 'frequency']]
quemas['frequency'] = pd.to_numeric(quemas['frequency'], errors='coerce').fillna(0)

print(f'  Polígonos de quemas: {len(quemas):,}')
# Spatial join: cada punto obtiene la frecuencia del polígono de quema donde cae
joined = gpd.sjoin(gdf[['geometry']], quemas, how='left', predicate='within')
freq   = joined['frequency'].groupby(level=0).max()  # max si cae en varios
gdf['quemas_freq'] = freq.reindex(gdf.index).fillna(0)
print(f'  Puntos con historial de quemas: {(gdf["quemas_freq"]>0).mean()*100:.1f}%')

Cargando quemas...
  Polígonos de quemas: 1,285,518
  Puntos con historial de quemas: 12.6%


## 4. Agregación a nivel especie

Cada especie pasa de N filas (ocurrencias) a 1 fila con features resumidas.

In [13]:
from scipy.stats import mode as scipy_mode

def safe_mode(x):
    """Moda robusta para categorías."""
    vals = x.dropna()
    if len(vals) == 0:
        return np.nan
    return vals.mode().iloc[0]

# Columnas de features continuas → media por especie
CONT_COLS = [
    'bio1', 'bio4', 'bio7', 'bio12', 'bio14', 'bio15',
    'forest_loss_total', 'forest_loss_recent',
    'anp_inside', 'anp_dist_km',
    'ti_inside', 'ti_dist_km',
    'min_ilegal_inside', 'min_ilegal_dist_km',
    'concesion_inside', 'concesion_dist_km',
    'petroleo_inside', 'petroleo_dist_km',
    'via_dist_km',
    'quemas_freq',
]

# Metadatos por especie
meta = gdf.groupby('species').agg(
    class_=('class', 'first'),
    iucn_categoria=('iucn_categoria', 'first'),
    n_occ=('gbifID', 'count'),
    lat_centroid=('decimalLatitude', 'mean'),
    lon_centroid=('decimalLongitude', 'mean'),
    year_min=('year', 'min'),
    year_max=('year', 'max'),
).rename(columns={'class_': 'class'})

# Features continuas → media
feat_mean = gdf.groupby('species')[CONT_COLS].mean()
# Renombrar _inside → pct_ para que sea más claro
feat_mean = feat_mean.rename(columns={
    'anp_inside':         'pct_anp',
    'ti_inside':          'pct_ti',
    'min_ilegal_inside':  'pct_min_ilegal',
    'concesion_inside':   'pct_concesion',
    'petroleo_inside':    'pct_petroleo',
    'forest_loss_total':  'pct_forest_loss_total',
    'forest_loss_recent': 'pct_forest_loss_recent',
})

# Cobertura de suelo → % de puntos en cada categoría
lc_dummies = pd.get_dummies(gdf.set_index('species')['lc_cat'], prefix='lc')
lc_pct     = lc_dummies.groupby(level=0).mean()

# Unir todo
species_df = meta.join(feat_mean).join(lc_pct)

print(f'DataFrame especie: {species_df.shape}')
print(f'Columnas: {list(species_df.columns)}')
species_df.head(3)

DataFrame especie: (1693, 32)
Columnas: ['class', 'iucn_categoria', 'n_occ', 'lat_centroid', 'lon_centroid', 'year_min', 'year_max', 'bio1', 'bio4', 'bio7', 'bio12', 'bio14', 'bio15', 'pct_forest_loss_total', 'pct_forest_loss_recent', 'pct_anp', 'anp_dist_km', 'pct_ti', 'ti_dist_km', 'pct_min_ilegal', 'min_ilegal_dist_km', 'pct_concesion', 'concesion_dist_km', 'pct_petroleo', 'petroleo_dist_km', 'via_dist_km', 'quemas_freq', 'lc_agriculture', 'lc_forest', 'lc_natural', 'lc_no_data', 'lc_water']


,class,iucn_categoria,n_occ,lat_centroid,lon_centroid,year_min,year_max,bio1,bio4,bio7,...,concesion_dist_km,pct_petroleo,petroleo_dist_km,via_dist_km,quemas_freq,lc_agriculture,lc_forest,lc_natural,lc_no_data,lc_water
species,,,,,,,,,,,,,,,,,,,,,
Abrothrix andinus,Mammalia,None,1,-17.168056,-69.395278,2003,2003,5.125573,208.983109,23.195749,...,160.928234,0.0,241.659451,151.200922,0.0,0.0,0.0,0.0,1.0,0.0
Abrothrix jelskii,Mammalia,LC,2,-16.236531,-68.191953,2017,2024,5.151948,160.726135,21.799500,...,8.781367,0.0,77.290849,18.135443,0.0,0.0,0.0,0.0,1.0,0.0
Accipiter bicolor,Aves,LC,2,-14.474833,-68.558166,2000,2000,20.647385,117.548706,15.686875,...,20.190049,0.5,13.056227,26.708312,0.0,0.0,0.0,0.0,1.0,0.0


## 5. Agregar rasgos biológicos (AVONET + PanTHERIA)

In [14]:
avonet = pd.read_csv(DATA / 'traits/avonet_slim.csv')
pantheria = pd.read_csv(DATA / 'traits/pantheria_slim.csv')

# Renombrar columnas PanTHERIA a nombres más cortos
pantheria = pantheria.rename(columns={
    'MSW05_Binomial':              'species',
    '5-1_AdultBodyMass_g':         'body_mass_g',
    '6-2_TrophicLevel':            'trophic_level',
    '1-1_ActivityCycle':           'activity_cycle',
    '26-1_GR_Area_km2':            'range_area_km2',
    '21-1_PopulationDensity_n/km2':'pop_density',
}).replace(-999, np.nan)

# Solo columnas útiles (descartar las >75% nulas)
pantheria = pantheria[['species', 'body_mass_g', 'trophic_level',
                        'activity_cycle', 'range_area_km2']]

# Renombrar AVONET
avonet = avonet.rename(columns={'Species1': 'species'})

# Join por clase
aves_idx = species_df[species_df['class'] == 'Aves'].index
mam_idx  = species_df[species_df['class'] == 'Mammalia'].index

species_df = species_df.join(
    avonet.set_index('species'), how='left'
).join(
    pantheria.set_index('species'), how='left', rsuffix='_mam'
)

# Verificar cobertura
av_match  = species_df.loc[aves_idx, 'Mass'].notna().sum()
mam_match = species_df.loc[mam_idx, 'body_mass_g'].notna().sum()
print(f'Aves con traits AVONET: {av_match}/{len(aves_idx)} ({av_match/len(aves_idx)*100:.1f}%)')
print(f'Mamíferos con traits PanTHERIA: {mam_match}/{len(mam_idx)} ({mam_match/len(mam_idx)*100:.1f}%)')

Aves con traits AVONET: 1315/1476 (89.1%)
Mamíferos con traits PanTHERIA: 163/217 (75.1%)


## 6. Target binario y filtros finales

In [15]:
# Filtro por minimo de ocurrencias
# Elimina especies con muy pocas observaciones en Bolivia (errores geograficos
# o visitantes accidentales no residentes).
MIN_OCC = 3
antes = len(species_df)
species_df = species_df[species_df['n_occ'] >= MIN_OCC]
print(f'Especies filtradas por n_occ < {MIN_OCC}: {antes - len(species_df)}')
print(f'Especies restantes: {len(species_df)}')

# Target binario: threatened = VU + EN + CR = 1, LC + NT = 0
species_df['threatened'] = species_df['iucn_categoria'].map({
    'LC': 0, 'NT': 0, 'VU': 1, 'EN': 1, 'CR': 1
})

# Reporte
for clase in ['Aves', 'Mammalia']:
    df_c = species_df[species_df['class'] == clase]
    con_label = df_c['threatened'].notna()
    n_thr = df_c.loc[con_label, 'threatened'].sum()
    n_tot = con_label.sum()
    print(f'{clase}: {len(df_c)} spp | con etiqueta IUCN: {n_tot} | '
          f'threatened: {int(n_thr)} ({n_thr/n_tot*100:.1f}%) | '
          f'not threatened: {int(n_tot - n_thr)}')

print(f'Total especies: {len(species_df)}')
print(f'Sin etiqueta IUCN (educated guesses target): {species_df["threatened"].isna().sum()}')


Especies filtradas por n_occ < 3: 145
Especies restantes: 1548
Aves: 1398 spp | con etiqueta IUCN: 1244 | threatened: 36 (2.9%) | not threatened: 1208
Mammalia: 150 spp | con etiqueta IUCN: 129 | threatened: 9 (7.0%) | not threatened: 120
Total especies: 1548
Sin etiqueta IUCN (educated guesses target): 175


## 7. Revisión de missing values

In [16]:
# Nulos por columna (solo features, no metadata)
feat_cols = [c for c in species_df.columns
             if c not in ['class', 'iucn_categoria', 'threatened',
                          'year_min', 'year_max']]

nulls = species_df[feat_cols].isnull().mean().sort_values(ascending=False)
nulls_pct = (nulls * 100).round(1)
print('Nulos por columna (% de especies):')
print(nulls_pct[nulls_pct > 0].to_string())

Nulos por columna (% de especies):
activity_cycle        95.2
trophic_level         94.0
body_mass_g           92.9
range_area_km2        92.8
Habitat               19.6
Migration             19.4
Beak.Length_Culmen    19.4
Beak.Width            19.4
Wing.Length           19.4
Kipps.Distance        19.4
Range.Size            19.4
Primary.Lifestyle     19.4
Beak.Depth            19.4
Tarsus.Length         19.4
Trophic.Niche         19.4
Trophic.Level         19.4
Mass                  19.4
Habitat.Density       19.4
bio1                   0.1
bio4                   0.1
bio15                  0.1
bio7                   0.1
bio12                  0.1
bio14                  0.1


## 8. Guardar

In [17]:
out_path = OUT / 'species_features.parquet'
species_df.to_parquet(out_path, engine='fastparquet')

print(f'Guardado: {out_path}')
print(f'Shape final: {species_df.shape}')
print(f'\nResumen columnas:')
print(species_df.dtypes.to_string())

Guardado: ../data/processed/species_features.parquet
Shape final: (1548, 51)

Resumen columnas:
class                      object
iucn_categoria             object
n_occ                       int64
lat_centroid              float64
lon_centroid              float64
year_min                    int64
year_max                    int64
bio1                      float32
bio4                      float32
bio7                      float32
bio12                     float32
bio14                     float32
bio15                     float32
pct_forest_loss_total     float64
pct_forest_loss_recent    float64
pct_anp                   float64
anp_dist_km               float64
pct_ti                    float64
ti_dist_km                float64
pct_min_ilegal            float64
min_ilegal_dist_km        float64
pct_concesion             float64
concesion_dist_km         float64
pct_petroleo              float64
petroleo_dist_km          float64
via_dist_km               float64
quemas_freq         